# 02 — Non-Destructive Data Cleaning & Grain-Safe Aggregation

Implements approved decisions **D-1 through D-5**:
- **D-1 (Revenue Proxy):** Delivered `payment_value` (`15,422,461.77 BRL` across `96,478` delivered orders) is the headline order-value proxy; `SUM(price)` is used for product, category, and seller analysis.
- **D-2 (Analysis Window):** `2017-01-01` to `2018-08-31` (`in_analysis_window = True`) for monthly trends and cohort retention.
- **D-3 (Reviews):** Most recent review per order used for order-level score.
- **D-4 (Category Translations):** Two manual mappings (`pc_gamer`, `portable_kitchen_and_food_preparers`) documented; unassigned mapped to `Unknown`.
- **D-5 (Repeat Customers):** `2+` delivered orders on `2+` distinct purchase dates using `customer_unique_id`.

In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
clean_log = json.loads((ROOT / "reports" / "data_quality" / "cleaning_log.json").read_text(encoding="utf-8"))

print("=== OUTPUT ROW COUNTS ===")
for k, v in clean_log["output_row_counts"].items():
    print(f"{k:<30s}: {v:>10,d}")

print("\n=== QUALITY FLAGS FLAGGED (NO SILENT DELETIONS) ===")
for k, v in clean_log["quality_flag_counts"].items():
    print(f"{k:<32s}: {v:>10,d}")

print("\n=== FINANCIAL RECONCILIATION (BRL) ===")
for k, v in clean_log["reconciliation"].items():
    print(f"{k:<32s}: R$ {v:>13,.2f}")

=== OUTPUT ROW COUNTS ===
orders_clean                  :     99,441
customers_clean               :     99,441
order_items_clean             :    112,650
payments_clean                :    103,886
reviews_clean                 :     99,224
products_clean                :     32,951
sellers_clean                 :      3,095
geolocation_zip_lookup        :     17,737
order_item_summary            :     98,666
order_payment_summary         :     99,440
order_review_summary          :     98,673
order_level                   :     99,441

=== QUALITY FLAGS FLAGGED (NO SILENT DELETIONS) ===
missing_delivery_date           :      2,965
missing_stage_date              :      2,980
carrier_before_purchase         :        166
carrier_before_approval         :      1,359
delivered_before_carrier        :         23
status_date_conflict            :          8
orders_without_items            :        775
shipping_limit_outliers         :         12
invalid_installments            :          2
